# Exact Diagonalization Data Exporter for Interactive Visualization

This notebook generates and exports all necessary physical observables of the XXZ Heisenberg chain to JSON files. The generated data is used directly by the Next.js frontend to visualize the quantum phase transition, local spin correlations, and the Schmidt entanglement spectrum in real-time.


## 1. Spin-Spin Correlations

To study the spatial structure of the phases, we calculate the longitudinal spin correlation function between adjacent sites:

$$C_z(i) = \langle \psi_{GS} | S_i^z S_{i+1}^z | \psi_{GS} \rangle$$

In the zero-magnetization sector ($S^z_{\text{tot}} = 0$), this can be calculated efficiently from state probabilities:

$$C_z(i) = \sum_{\{\sigma\}} |\langle \sigma | \psi_{GS} \rangle|^2 \sigma_i \sigma_{i+1}$$

where $\sigma_i = \pm 1/2$ is the spin state at site $i$ (i.e. $\sigma_i = 1$ if the $i$-th bit is $1$, else $-1$, in units where $S^z = \sigma^z / 2$, so $C_z(i) = \frac{1}{4} \sigma_i \sigma_{i+1}$).


In [3]:
import os
import json
import numpy as np
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigsh
from math import comb

# Output directory for Next.js public assets
OUT = "/home/ayush/Desktop/code/exact_diagonalization/frontend/public"
os.makedirs(OUT, exist_ok=True)
print(f"Export directory: {OUT}")


Export directory: /home/ayush/Desktop/code/exact_diagonalization/frontend/public


In [4]:
def generate_sz0_basis(N):
    basis = []
    for state in range(1 << N):
        if bin(state).count('1') == N // 2:
            basis.append(state)
    return basis

def state_to_spins(state, N):
    return [(1 if (state >> i) & 1 else -1) for i in range(N)]

def build_hamiltonian(N, J, Delta, pbc=True):
    basis = generate_sz0_basis(N)
    dim = len(basis)
    state_to_idx = {s: i for i, s in enumerate(basis)}
    H = lil_matrix((dim, dim))
    for idx, state in enumerate(basis):
        diag = 0.0
        for i in range(N):
            j = (i + 1) % N if pbc else i + 1
            if j >= N: continue
            si, sj = (state >> i) & 1, (state >> j) & 1
            diag += 1.0 if si == sj else -1.0
        H[idx, idx] = J * Delta * diag / 4.0
        for i in range(N):
            j = (i + 1) % N if pbc else i + 1
            if j >= N: continue
            si, sj = (state >> i) & 1, (state >> j) & 1
            if si == 1 and sj == 0:
                ns = state & ~(1 << i); ns |= 1 << j
                H[idx, state_to_idx[ns]] += J / 2.0
            elif si == 0 and sj == 1:
                ns = state | (1 << i); ns &= ~(1 << j)
                H[idx, state_to_idx[ns]] += J / 2.0
    return H.tocsr(), basis

def entanglement_entropy(psi, basis, N, k=None):
    if k is None: k = N // 2
    dimA, dimB = 1 << k, 1 << (N - k)
    M = np.zeros((dimA, dimB))
    for coeff, state in zip(psi, basis):
        a = state & ((1 << k) - 1); b = state >> k
        M[a, b] = coeff
    _, s, _ = np.linalg.svd(M, full_matrices=False)
    ssq = s[s > 1e-15]**2; ssq /= ssq.sum()
    return -np.sum(ssq * np.log(ssq))

def compute_correlations(psi, basis, N):
    sz_sz = np.zeros(N)
    prob = np.abs(psi)**2
    for amp, state in zip(prob, basis):
        spins = state_to_spins(state, N)
        for i in range(N - 1):
            sz_sz[i] += amp * spins[i] * spins[i+1]
    return sz_sz.tolist()


## 2. Generating and Exporting Data


In [5]:
J = 1.0
N = 10

# 2.1 Energy spectrum (N=10, multiple Δ)
print("=== 1. Energy spectrum ===")
Delta_range = np.linspace(0.2, 2.0, 37)
n_states = 6
spectrum = []
for Delta in Delta_range:
    H, basis = build_hamiltonian(N, J, Delta)
    evals = eigsh(H, k=n_states, which='SA', return_eigenvectors=False)
    spectrum.append({"Delta": round(Delta, 4), "energies": [round(e, 6) for e in evals.tolist()]})
with open(f"{OUT}/energy_spectrum.json", "w") as f:
    json.dump(spectrum, f)
print(f"Saved energy_spectrum.json ({len(spectrum)} points)")

# 2.2 Phase transition (N=10, entropy + gap)
print("=== 2. Phase transition scan ===")
Delta_range_fine = np.linspace(0.2, 2.0, 55)
phase = []
for Delta in Delta_range_fine:
    H, basis = build_hamiltonian(N, J, Delta)
    evals, evecs = eigsh(H, k=3, which='SA')
    psi = evecs[:, 0]
    S = entanglement_entropy(psi, basis, N)
    phase.append({"Delta": round(Delta, 4), "entropy": round(S, 6), "gap": round(evals[1] - evals[0], 6), "E0": round(evals[0], 6)})
with open(f"{OUT}/phase_transition.json", "w") as f:
    json.dump(phase, f)
print(f"Saved phase_transition.json ({len(phase)} points)")

# 2.3 Size scaling (N=8,10,12)
print("=== 3. Size scaling ===")
sizes = [8, 10, 12]
Delta_coarse = np.linspace(0.2, 2.0, 28)
scaling = []
for Delta in Delta_coarse:
    row = {"Delta": round(Delta, 4)}
    for Nc in sizes:
        H, basis = build_hamiltonian(Nc, J, Delta)
        evals, evecs = eigsh(H, k=2, which='SA')
        S = entanglement_entropy(evecs[:, 0], basis, Nc)
        row[f"S_N{Nc}"] = round(S, 6)
        row[f"dim_N{Nc}"] = comb(Nc, Nc // 2)
    scaling.append(row)
with open(f"{OUT}/size_scaling.json", "w") as f:
    json.dump(scaling, f)
print(f"Saved size_scaling.json ({len(scaling)} points)")

# 2.4 Spin chain snapshots (N=10, various Δ)
print("=== 4. Spin chain snapshots ===")
chain_data = []
for Delta in [0.2, 0.5, 1.0, 1.5, 2.0]:
    H, basis = build_hamiltonian(N, J, Delta)
    evals, evecs = eigsh(H, k=1, which='SA')
    psi = evecs[:, 0]
    prob = np.abs(psi)**2
    top_idx = np.argsort(prob)[-5:][::-1]
    top_configs = []
    for idx in top_idx:
        spins = state_to_spins(basis[idx], N)
        top_configs.append({"spins": spins, "prob": round(float(prob[idx]), 6)})
    corr = compute_correlations(psi, basis, N)
    chain_data.append({"Delta": round(Delta, 4), "E0": round(float(evals[0]), 6), "top_configs": top_configs, "corr": corr})
with open(f"{OUT}/spin_chain.json", "w") as f:
    json.dump(chain_data, f)
print(f"Saved spin_chain.json ({len(chain_data)} snapshots)")

# 2.5 Entanglement spectrum (N=10, Δ=1)
print("=== 5. Entanglement spectrum ===")
Nc = 10
H, basis = build_hamiltonian(Nc, J, 1.0)
evals, evecs = eigsh(H, k=1, which='SA')
psi = evecs[:, 0]
dimA = 1 << (Nc // 2); dimB = 1 << (Nc - Nc // 2)
M = np.zeros((dimA, dimB))
for coeff, state in zip(psi, basis):
    a = state & ((1 << (Nc // 2)) - 1); b = state >> (Nc // 2)
    M[a, b] = coeff
_, s, _ = np.linalg.svd(M, full_matrices=False)
ssq = s[s > 1e-15]**2; ssq = ssq / ssq.sum() if ssq.sum() > 0 else ssq
ent_spec = [round(float(v), 8) for v in sorted(ssq, reverse=True)[:20]]
with open(f"{OUT}/entanglement_spectrum.json", "w") as f:
    json.dump(ent_spec, f)
print(f"Saved entanglement_spectrum.json (top 20 Schmidt values)")


=== 1. Energy spectrum ===
Saved energy_spectrum.json (37 points)
=== 2. Phase transition scan ===
Saved phase_transition.json (55 points)
=== 3. Size scaling ===
Saved size_scaling.json (28 points)
=== 4. Spin chain snapshots ===
Saved spin_chain.json (5 snapshots)
=== 5. Entanglement spectrum ===
Saved entanglement_spectrum.json (top 20 Schmidt values)


## 3. Verify Generated Data files


In [6]:
import glob
json_files = glob.glob(f"{OUT}/*.json")
print("Verified files in public folder:")
for path in sorted(json_files):
    name = os.path.basename(path)
    size = os.path.getsize(path)
    with open(path, 'r') as f:
        data = json.load(f)
    print(f" - {name}: {size} bytes, type={type(data)}, elements={len(data) if isinstance(data, (list, dict)) else 'N/A'}")


Verified files in public folder:
 - energy_spectrum.json: 3536 bytes, type=<class 'list'>, elements=37
 - entanglement_spectrum.json: 230 bytes, type=<class 'list'>, elements=20
 - phase_transition.json: 3992 bytes, type=<class 'list'>, elements=55
 - size_scaling.json: 3354 bytes, type=<class 'list'>, elements=28
 - spin_chain.json: 2915 bytes, type=<class 'list'>, elements=5
